# Week 7 of learning python with TDI (The Data Immersed)


In [ ]:
#Data Transformation and Featured Engineering
#Week 7 is all about enriching your dataset. Using Pandas, you handle missing values, convert data types, clean strings, 
#and create new columns that reveal hidden patterns. 
#This is where raw data becomes meaningful insights, ready for deeper analysis.

In [1]:
#Task 1:  Handle Missing Values 

import pandas as pd

# Load your dataset
df = pd.read_csv("supply_chain.csv")

# 1. Find which columns have missing values
missing_columns = df.columns[df.isnull().any()]
print("Columns with missing values:", missing_columns.tolist())

# 2. Count how many missing values in each column
missing_counts = df.isnull().sum()
print("Missing values per column:\n", missing_counts)

# 3. Handle columns with few missing values
for col in df.columns:
    if df[col].isnull().sum() > 0:
        # Numeric columns → fill with mean
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col].fillna(df[col].mean(), inplace=True)
        # Text columns → fill with most common value (mode)
        elif pd.api.types.is_string_dtype(df[col]):
            df[col].fillna(df[col].mode()[0], inplace=True)

# 4. Handle columns with many missing values
# Example: drop rows if more than 50% of values in a row are missing
threshold = len(df.columns) / 2
df = df.dropna(thresh=threshold)

# Alternatively, drop rows where a specific important column is missing
# df = df.dropna(subset=['important_col'])

# 5. Verify missing values are gone or handled
print("Remaining missing values:\n", df.isnull().sum())


Columns with missing values: ['product_defects', 'return_status']
Missing values per column:
 order_id                   0
order_date                 0
product_category           0
quantity                   0
unit_price                 0
total_cost                 0
supplier_name              0
supplier_location          0
lead_time_days             0
warehouse_destination      0
delivery_date              0
actual_delivery_days       0
delivery_status            0
on_time                    0
product_defects          154
return_status            146
dtype: int64
Remaining missing values:
 order_id                   0
order_date                 0
product_category           0
quantity                   0
unit_price                 0
total_cost                 0
supplier_name              0
supplier_location          0
lead_time_days             0
warehouse_destination      0
delivery_date              0
actual_delivery_days       0
delivery_status            0
on_time                  

In [5]:
#Task 2:Convert Data Types 
import pandas as pd

# Load your dataset
df = pd.read_csv("supply_chain.csv")

# 1. Check the current data type of each column
print("Data types before conversion:\n", df.dtypes)

# 2. Convert date columns to datetime
# Example: if you have 'order_date' and 'delivery_date'
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df['delivery_date'] = pd.to_datetime(df['delivery_date'], errors='coerce')

# 3. Convert numeric text to actual numbers
# Example: if you have 'cost' or 'quantity' stored as text
df['total_cost'] = pd.to_numeric(df['total_cost'], errors='coerce')
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')

# 4. Check your work: print the data types to confirm changes
print("Data types after conversion:\n", df.dtypes)



Data types before conversion:
 order_id                  object
order_date                object
product_category          object
quantity                   int64
unit_price               float64
total_cost               float64
supplier_name             object
supplier_location         object
lead_time_days             int64
warehouse_destination     object
delivery_date             object
actual_delivery_days       int64
delivery_status           object
on_time                   object
product_defects          float64
return_status             object
dtype: object
Data types after conversion:
 order_id                         object
order_date               datetime64[ns]
product_category                 object
quantity                          int64
unit_price                      float64
total_cost                      float64
supplier_name                    object
supplier_location                object
lead_time_days                    int64
warehouse_destination            obje

In [6]:
#Task 3: Create Simple Calculated Columns for 3 sections
import pandas as pd

# Load your dataset
df = pd.read_csv("supply_chain.csv")

# --- Column 1: Days to Deliver ---
# Make sure order_date and delivery_date are datetime first
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df['delivery_date'] = pd.to_datetime(df['delivery_date'], errors='coerce')

# Calculate difference in days
df['days_to_deliver'] = (df['delivery_date'] - df['order_date']).dt.days

# --- Column 2: Order Size Category ---
def categorize_order_size(qty):
    if qty < 50:
        return "Small"
    elif 50 <= qty <= 200:
        return "Medium"
    else:
        return "Large"

df['order_size_category'] = df['quantity'].apply(categorize_order_size)

# --- Column 3: High Cost Flag ---
avg_cost = df['total_cost'].mean()
df['high_cost_flag'] = df['total_cost'].apply(lambda x: 1 if x > avg_cost else 0)

# --- Display sample data with new columns ---
print(df[['order_date', 'delivery_date', 'days_to_deliver',
          'quantity', 'order_size_category',
          'total_cost', 'high_cost_flag']].head())


  order_date delivery_date  days_to_deliver  quantity order_size_category  \
0 2022-11-28    2022-12-30               32        94              Medium   
1 2022-02-20    2022-03-09               17        16               Small   
2 2022-11-03    2022-11-20               17        25               Small   
3 2022-12-15    2023-01-20               36        33               Small   
4 2022-03-02    2022-03-17               15        13               Small   

   total_cost  high_cost_flag  
0     6062.06               0  
1     1226.08               0  
2    53976.50               0  
3    25563.12               0  
4    27188.33               0  


In [8]:
#task 4:Clean the Text Data

import pandas as pd

# Load your dataset
df = pd.read_csv("supply_chain.csv")

# --- Display before cleaning ---
print("Before cleaning:\n", df[['supplier_name', 'product_category']].head())

# 1. Remove extra spaces
df['supplier_name'] = df['supplier_name'].str.strip()
df['product_category'] = df['product_category'].str.strip()

# 2. Standardize capitalization (choose one: title case or uppercase)
df['supplier_name'] = df['supplier_name'].str.title()   # Title Case
# Alternative: df['supplier_name'] = df['supplier_name'].str.upper()

# 3. Fix spelling inconsistencies
# Example: replace "Moter" with "Motor"
df['supplier_name'] = df['supplier_name'].replace({'Moter': 'Motor'})
df['product_category'] = df['product_category'].replace({'Moter': 'Motor'})

# 4. Display after cleaning
print("After cleaning:\n", df[['supplier_name', 'product_category']].head())


Before cleaning:
   supplier_name product_category
0     SupplierE        Fasteners
1    Supplier E        Fasteners
2    Supplier A  Control Systems
3     supplierc           Valves
4     supplierb           Motors
After cleaning:
   supplier_name product_category
0     Suppliere        Fasteners
1    Supplier E        Fasteners
2    Supplier A  Control Systems
3     Supplierc           Valves
4     Supplierb           Motors


In [10]:
#Task 5: Create Performance Metrics 

import pandas as pd

# Load dataset
df = pd.read_csv("supply_chain.csv")

# Make sure order_date and delivery_date are datetime
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df['delivery_date'] = pd.to_datetime(df['delivery_date'], errors='coerce')

# --- Step 1: Create 'days_to_deliver' first ---
df['days_to_deliver'] = (df['delivery_date'] - df['order_date']).dt.days

# --- Column 1: On Time (1 or 0) ---
df['on_time'] = df.apply(
    lambda row: 1 if row['days_to_deliver'] <= row['lead_time_days'] else 0,
    axis=1
)

# --- Column 2: Cost Per Unit ---
df['cost_per_unit'] = df['total_cost'] / df['quantity']

# --- Column 3: Supplier Performance Grade ---
supplier_perf = df.groupby('supplier_name')['on_time'].mean().reset_index()
supplier_perf.rename(columns={'on_time': 'supplier_performance_grade'}, inplace=True)

# Merge back into main DataFrame
df = df.merge(supplier_perf, on='supplier_name', how='left')

# --- Display sample data with performance metrics ---
print(df[['supplier_name', 'order_date', 'delivery_date',
          'days_to_deliver', 'lead_time_days',
          'on_time', 'quantity', 'cost_per_unit',
          'supplier_performance_grade']].head())



  supplier_name order_date delivery_date  days_to_deliver  lead_time_days  \
0     SupplierE 2022-11-28    2022-12-30               32              32   
1    Supplier E 2022-02-20    2022-03-09               17              17   
2    Supplier A 2022-11-03    2022-11-20               17              17   
3     supplierc 2022-12-15    2023-01-20               36              36   
4     supplierb 2022-03-02    2022-03-17               15              15   

   on_time  quantity  cost_per_unit  supplier_performance_grade  
0        1        94          64.49                    0.773663  
1        1        16          76.63                    0.740541  
2        1        25        2159.06                    0.954373  
3        1        33         774.64                    0.858065  
4        1        13        2091.41                    0.872146  


In [ ]:
#Such an exciting week diving into Data Transformation & Feature Engineering!
#Handling missing values, converting data types, cleaning strings, and creating new features made the dataset richer 
#and more insightful.
#Looking forward to Week 8 and exploring GroupBy operations to uncover patterns across suppliers and segments!